# Projeto: 7 Dias de Código
## Importação de bibliotecas / Configuração de parâmetros

In [1]:
from pathlib import Path
from typing  import Union, Optional
#
import logging as log
import numpy   as np
import os
import pandas  as pd
#
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 30
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

## Funções Genéricas do Projeto

In [2]:

def fxPathFile(strPathFile: str, strFile: str, strExtFile: str) -> str:
    """
    Função..: fxPathFile
    Objetivo: Concatena o Caminho e o Nome do arquivo em uma única variável
            Caso o diretório não exista, ele será criado.
    """
    if not strPathFile:
        raise ValueError("O caminho base não pode ser vazio.")

    if not strFile:
        raise ValueError("O nome do arquivo não pode ser vazio.")

    # Cria o diretório caso não exista
    os.makedirs(strPathFile, exist_ok=True)

    # Garante extensão do arquivo
    if strExtFile and not strFile.lower().endswith(f".{strExtFile}"):
        strFile = f"{strFile}.{strExtFile}"

    return os.path.join(strPathFile, strFile)

def fxOpenFile(strFile: str):
    """
    Função..: fxOpenFile
    Objetivo: Abrir arquivos tipo parquet.
            Em caso de falha na abertura, retorna uma mensagem de erro específica para OSError
            e imprime a mensagem de erro para outros tipos de falha.
    """
    try:
        df = pd.read_parquet(strFile)
        return df
    except OSError as e:
        print(f"ERRO: Falha ao abrir o arquivo '{strFile}'. Verifique o caminho ou a integridade do arquivo. Detalhes: {e}")
        return None
    except Exception as e:
        print(f"ERRO: Ocorreu um erro inesperado ao abrir o arquivo '{strFile}'. Detalhes: {e}")
        return None

def fxSaveParquet(
    df: pd.DataFrame,
    caminho: Union[str, Path],
    verificar_integridade: bool = True,
    comprimir: Optional[str] = 'snappy'
) -> bool:
    """
    Função..: fxSaveParquet
    Objetivo: Salva um DataFrame em formato Parquet e verifica a gravação.
    Args....:
        df: DataFrame a ser salvo
        caminho: Caminho completo do arquivo (incluindo .parquet)
        verificar_integridade: Se True, valida a gravação lendo o arquivo
        comprimir: Tipo de compressão ('snappy', 'gzip', 'brotli' ou None)

    Returns:
        bool: True se gravação bem-sucedida, False caso contrário
    """
    # Validação inicial: DataFrame não pode estar vazio
    if df.empty:
        raise ValueError("DataFrame não pode estar vazio")

    # Converte para Path para manipulação consistente de caminhos
    caminho_arquivo = Path(caminho)

    # Garante que o arquivo tenha extensão .parquet
    if caminho_arquivo.suffix != '.parquet':
        caminho_arquivo = caminho_arquivo.with_suffix('.parquet')

    try:
        # Cria diretórios intermediários se não existirem
        caminho_arquivo.parent.mkdir(parents=True, exist_ok=True)

        # Salva o DataFrame em formato Parquet
        df.to_parquet(
            caminho_arquivo,
            compression=comprimir,
            index=False,
            engine='pyarrow'  # Engine padrão no Colab
        )

        # Verifica se o arquivo foi realmente criado no sistema
        if not caminho_arquivo.exists():
            log.error(f"Arquivo não foi criado: {caminho_arquivo}")
            return False

        # Validação de integridade: lê o arquivo e compara com original
        if verificar_integridade:
            df_validacao = pd.read_parquet(caminho_arquivo)

            # Verifica se dimensões (linhas x colunas) são iguais
            if df.shape != df_validacao.shape:
                log.error("Dimensões do arquivo salvo diferem do original")
                return False

            # Verifica se os nomes das colunas correspondem
            if not df.columns.equals(df_validacao.columns):
                log.error("Colunas do arquivo salvo diferem do original")
                return False

        # Exibe tamanho do arquivo salvo
        tamanho_mb = caminho_arquivo.stat().st_size / (1024 * 1024)
        log.info(f"✓ Arquivo salvo: {caminho_arquivo} ({tamanho_mb:.2f} MB)")

        return True

    except PermissionError:
        log.error(f"Sem permissão para escrever em: {caminho_arquivo}")
        return False

    except Exception as e:
        log.error(f"Erro ao salvar arquivo: {str(e)}")
        return False

def fxSaveDownload(
    df: pd.DataFrame,
    nome_arquivo: str = 'dados.parquet',
    baixar: bool = True
) -> bool:
    """
    Salva Parquet e oferece download automático no Colab.

    Args:
        df: DataFrame a ser salvo
        nome_arquivo: Nome do arquivo (sem necessidade de caminho)
        baixar: Se True, inicia download automaticamente

    Returns:
        bool: True se operação bem-sucedida
    """
    # Salva o arquivo no diretório temporário do Colab
    sucesso = fxSaveParquet(df, nome_arquivo)

    if sucesso and baixar:
        try:
            # Faz download do arquivo para máquina local
            files.download(nome_arquivo)
            log.info(f"Download iniciado: {nome_arquivo}")
        except Exception as e:
            log.error(f"Erro ao fazer download: {str(e)}")
            return False

    return sucesso


def fxRemoveFileIfExists(strPathFile: str) -> None:
    """
    Função..: fxRemoveFileIfExists
    Objetivo: Remover o arquivo caso ele exista.
    """
    if os.path.isfile(strPathFile):
        os.remove(strPathFile)

def fxAcertaDataHora(df, strNomeColuna):
    """
    Função - fxAcertaDataHora
    Objetivo.: Transformar as colunas: [data_emprestimo, data_devolucao, data_renovacao'] que possuem o
            formato: YYYY-mm-dd HH:MM:SS.ffffffff para o formato: YYYY-mm-dd HH:MM
    """
    intContReg  = 1
    lstDataHora = []
    #
    for conteudo in df[strNomeColuna]:
        if str(conteudo) == 'nan':
            conteudo = ''
        elif len(conteudo) > 16:
            conteudo = conteudo[:16]
        #
        lstDataHora.append(conteudo)
        print(f'Registro: {str(intContReg)}, Data Devolucao: {conteudo}')
        intContReg += 1
    #
    df[strNomeColuna] = lstDataHora
    return None

"""
# Função: Converte os valores de uma coluna STRING de um DataFrame para Data
"""
fxConvParaData = lambda df, strNomeColuna : pd.to_datetime(df[strNomeColuna])

"""
# Função: Converte o valor de uma Variável STRING para Data
"""
fxConvStrParaData = lambda strNomeColuna : pd.to_datetime(strNomeColuna, dayfirst=True)


## 1ª Etapa - Importação dos dados

## 1ª Etapa - Parte 1 - Definição de todas as Constantes

In [ ]:
class Config:

    # Informações dos arquivos
    SEP_CSV     = ','
    EXT_CSV     = 'csv'
    EXT_EXCEL   = 'xlsx'
    EXT_JSON    = 'json'
    EXT_PARQUET = 'parquet'

    PREFIXO_ARQ_CSV = 'emprestimos-'

    # Nomes dos bancos de dados
    DB_EMPR_BRONZE = 'DB_Empr_Bronze'
    DB_EMPR_SILVER = 'DB_Empr_Silver'
    DB_EMPR_SILVER_V2 = 'DB_Empr_Silver_V2'
    DB_EMPR_GOLD   = 'DB_Empr_Gold'
    DB_EMPR_DATA   = 'DB_Empr_Data'
    DB_EMPR_ANO    = 'DB_Empr_Ano'
    DB_EMPR_MES    = 'DB_Empr_Mes'
    DB_EMPR_HORA   = 'DB_Empr_Hora'
    DB_EMPR_DUPLICADOS    = 'DB_Empr_Duplicados'
    DB_EMPR_INCONSISTENTE = 'DB_Empr_Inconsistente'
    DB_EMPR_PERDIDOS       = 'DB_Empr_Perdidos'

    # Caminhos e URLs
    PATH_BASE   :f'D:/Users/rtoni/OneDrive/Git-Dados/7DaysOfCode'
    URL_CSV     = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/'
    URL_PARQUET = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/raw/main/Dia_1-Importando_dados/Datasets/dados_exemplares.parquet'
    URL_EXCEL   = 'https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/raw/Dia_6-Novos_dados_novas_analises/Datasets/matricula_alunos.xlsx'

    # Parâmetros
    ANO_INICIAL  = 2010
    ANO_FINAL    = 2021
    ARQS_POR_ANO = 2

## 1ª Etapa - Parte 2 - Concatenando todos os arquivos CSV

In [4]:
dicNomeDF = dict()
for intAno in range(2010, 2021):       # Faixa de Anos que serão analisados: 2010 - 2020
    for intQtdArqs in range(1, 3):     # Quantidade de arquivos por Ano = 2
        """
        # dicNomeDF  - Dicionário que armazena temporariamente os dados dos arquivos
        # strNomeDF  - Nome do DataFrame atual
        # strURL_Arq - Endereço da URL do GIT onde estão os dados
        """
        strNomeDF   = f'df_{str(intAno)}_{str(intQtdArqs)}'
        strURL_File = f'{Config.URL_CSV}{Config.PREFIXO_ARQ_CSV}{str(intAno)}{str(intQtdArqs)}.{Config.EXT_CSV}?raw=true'
        print(strURL_File)
        try:
            """
            # Processo de tratamento dos dados Livros Emprestados:
            # - Converte as colunas que contém Data/Hora para Datatime
            # - Converte as colunas ['id_emprestimo'] e [matricula_e_siape] para String
            """
            dicNomeDF[strNomeDF] = pd.read_csv(strURL_File, sep=Config.SEP_CSV)
        except Exception as e:
            print(strNomeDF, e)
            continue

print(dicNomeDF.values)

https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20101.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20102.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20111.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20112.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20121.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura-Python-Pandas/blob/main/Dia_1-Importando_dados/Datasets/dados_emprestimos/emprestimos-20122.csv?raw=true
https://github.com/FranciscoFoz/7_Days_of_Code_Alura

## 1ª Etapa - Parte 3 - Unificar todos os dicionários em um único arquivo. Grava o arquivo em disco

In [5]:
bolResult     = False
lstDataFrames = []         # Lista que contém o(s) DataFrame(s) temporariamente
try:
    lstDataFrames.extend(dicNomeDF.values())
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_BRONZE, Config.EXT_PARQUET)
    df_Empr_Bronze = pd.concat(lstDataFrames, ignore_index = True)
    bolResult = fxSaveParquet(
        df = df_Empr_Bronze,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_BRONZE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )

except OSError as e:
    print(e)

Arquivo: DB_Empr_Bronze.parquet gravado com sucesso!!!


In [6]:
# Verificar se a identificação do empréstimo 2560028 está presente no DataFrame, conforme mostrado no arquivo de resolução do problema, disponibilizado pela Alura.
# Realizei este passo antes de eliminar os registros duplicados para confirmar a presença do mesmo no arquivo original, porém, não consegui encontrar o mesmo.
# Na solução disponibilizada, o mesmo está presente.

df_Empr_Bronze.loc[df_Empr_Bronze['id_emprestimo']==2560028]

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario


In [7]:
df_Empr_Bronze.head()

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario
0,709684,L095049,NaN,2010/01/04 07:44:10.721000000,2010/01/05 16:26:12.662000000,2.008023e+09,ALUNO DE GRADUAÇÃO
1,709685,L167050,NaN,2010/01/04 07:44:10.750000000,2010/01/12 07:34:13.934000000,2.008023e+09,ALUNO DE GRADUAÇÃO
2,709686,2006017618,2010/01/26 08:07:01.738000000,2010/01/04 08:08:44.081000000,2010/02/25 07:36:25.800000000,2.008112e+09,ALUNO DE PÓS-GRADUAÇÃO
3,709687,L184117,2010/01/18 11:07:46.470000000,2010/01/04 08:24:21.284000000,2010/02/03 08:58:45.692000000,2.007211e+08,ALUNO DE GRADUAÇÃO
4,709684,L095049,NaN,2010/01/04 07:44:10.721000000,2010/01/05 16:26:12.662000000,2.008023e+09,ALUNO DE GRADUAÇÃO


In [8]:
# Contagem de registros
intQtdTotReg  = len(df_Empr_Bronze)
intQtdRegUni  = len(df_Empr_Bronze.value_counts())

print(f'Qtd Total de registros...........: {intQtdTotReg:,} \n')
print(f'Qtd total de registros únicos....: {intQtdRegUni:,} \n')

Qtd Total de registros...........: 2,258,018 

Qtd total de registros únicos....: 968,028 



In [9]:
bolResult = False
df_Empr_Duplicados = df_Empr_Bronze[df_Empr_Bronze.duplicated(keep=False)].sort_values(by=df_Empr_Bronze.columns.tolist()).reset_index(drop=True)

try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_DUPLICADOS, Config.EXT_PARQUET)
    bolResult = fxSaveParquet(
        df = df_Empr_Duplicados,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_DUPLICADOS}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
except OSError as e:
    print(e)

Arquivo: DB_Empr_Duplicados.parquet gravado com sucesso!!!


In [10]:
print(f'Total de registros duplicados: {df_Empr_Bronze.duplicated().sum():,} no arquivo: {Config.DB_EMPR_BRONZE}.{Config.EXT_PARQUET} \n')
print(f'Total de registros duplicados: {len(df_Empr_Duplicados):,} no arquivo: {Config.DB_EMPR_DUPLICADOS}.{Config.EXT_PARQUET}')

Total de registros duplicados: 37 no arquivo: DB_Empr_Bronze.parquet 

Total de registros duplicados: 63 no arquivo: DB_Empr_Duplicados.parquet


In [11]:
df_Empr_Duplicados

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario
0,709684,L095049,NaN,2010/01/04 07:44:10.721000000,2010/01/05 16:26:12.662000000,2.008023e+09,ALUNO DE GRADUAÇÃO
1,709684,L095049,NaN,2010/01/04 07:44:10.721000000,2010/01/05 16:26:12.662000000,2.008023e+09,ALUNO DE GRADUAÇÃO
2,709685,L167050,NaN,2010/01/04 07:44:10.750000000,2010/01/12 07:34:13.934000000,2.008023e+09,ALUNO DE GRADUAÇÃO
3,709685,L167050,NaN,2010/01/04 07:44:10.750000000,2010/01/12 07:34:13.934000000,2.008023e+09,ALUNO DE GRADUAÇÃO
4,709686,2006017618,2010/01/26 08:07:01.738000000,2010/01/04 08:08:44.081000000,2010/02/25 07:36:25.800000000,2.008112e+09,ALUNO DE PÓS-GRADUAÇÃO
5,709686,2006017618,2010/01/26 08:07:01.738000000,2010/01/04 08:08:44.081000000,2010/02/25 07:36:25.800000000,2.008112e+09,ALUNO DE PÓS-GRADUAÇÃO
6,709687,L184117,2010/01/18 11:07:46.470000000,2010/01/04 08:24:21.284000000,2010/02/03 08:58:45.692000000,2.007211e+08,ALUNO DE GRADUAÇÃO
7,709687,L184117,2010/01/18 11:07:46.470000000,2010/01/04 08:24:21.284000000,2010/02/03 08:58:45.692000000,2.007211e+08,ALUNO DE GRADUAÇÃO
8,709698,2009047725,2010/01/18 14:44:41.163000000,2010/01/04 09:21:19.099000000,2010/02/02 12:02:38.444000000,2.009047e+09,ALUNO DE GRADUAÇÃO
9,709698,2009047725,2010/01/18 14:44:41.163000000,2010/01/04 09:21:19.099000000,2010/02/02 12:02:38.444000000,2.009047e+09,ALUNO DE GRADUAÇÃO


### Registros que não possuem Número de matrícula ou SIAPE

In [12]:
# Importante: Não foi encontrado nenhum registro que a coluna [data_emprestimo] ou [data_devolucao] possua valor nulo (NaN)

bolResult = False
try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_INCONSISTENTE, Config.EXT_PARQUET)
    df_Empr_Inconsistente = df_Empr_Bronze[df_Empr_Bronze['matricula_ou_siape'].isna()]
    #
    if len(df_Empr_Inconsistente) > 0:
        bolResult = fxSaveParquet(
        df = df_Empr_Inconsistente,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )

    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_INCONSISTENTE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
    print(f'Qtd Total de registros...........: {len(df_Empr_Inconsistente)}')

except OSError as e:
    print(e)

# Comando para selecionar os registros inconsistentes
# Quando uma ou mais colunas possuírem valores nulos (NaN) deve-se utilizar o operador OR (|)

# df_Inconsistente = df_Empr_Bronze[
    # df_Empr_Bronze['matricula_ou_siape'].isna()] |
    # df_Empr_Bronze['data_emprestimo'].isna()     |
    # df_Empr_Bronze['codigo_barras'].isna()
# ]

Arquivo: DB_Empr_Inconsistente.parquet gravado com sucesso!!!
Qtd Total de registros...........: 3170


In [13]:
df_Empr_Inconsistente.head()

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario
3482,713154,2007015558,2010/02/05 17:22:18.059000000,2010/01/21 11:38:42.615000000,2010/02/09 14:39:37.772000000,NaN,SERVIDOR TÉCNICO-ADMINISTRATIVO
4372,714044,L188570,2010/02/24 10:49:34.210000000,2010/01/26 15:08:42.587000000,2010/03/25 14:56:39.205000000,NaN,USUÁRIO EXTERNO
4375,714047,L148514,2010/02/24 10:49:34.193000000,2010/01/26 15:13:40.173000000,2010/03/25 14:56:25.572000000,NaN,USUÁRIO EXTERNO
4376,714048,L187527,2010/02/24 10:49:34.159000000,2010/01/26 15:13:40.204000000,2010/03/25 14:56:48.598000000,NaN,USUÁRIO EXTERNO
4537,714209,L165158,2010/02/05 17:22:18.034000000,2010/01/27 11:07:39.973000000,2010/02/08 09:49:36.146000000,NaN,SERVIDOR TÉCNICO-ADMINISTRATIVO


## 1ª Etapa - Parte 4 - Tratando Dataframe: df_Empr_Bronze
- Realizar a contagem dos registros
- Eliminar os registros duplicados, mantendo a primeira ocorrência
- Resultado gerar o df_Empr_Silver

In [14]:
# Eliminar registros duplicados
bolResult = False
try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_SILVER, Config.EXT_PARQUET)
    #
    # Verifica a quantidade total de registros duplicados e os apaga
    df_Empr_Silver = df_Empr_Bronze.drop_duplicates(
        # subset = [
            # 'matricula_ou_siape',
            # 'data_emprestimo',
            # 'data_devolucao'
        # ],
        keep = 'first'
    )

    bolResult = fxSaveParquet(
        df = df_Empr_Silver,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_SILVER}.{Config.EXT_PARQUET} gravado com sucesso!!! \n' )
        print(f'Total de registros após tratamento de dados.......: {len(df_Empr_Silver):,} \n')
        print(f'Total de registros únicos após tratamento de dados: {len(df_Empr_Silver.value_counts()):,}')

except OSError as e:
    print(e)


try:
    strPathFile = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_INCONSISTENTE, Config.EXT_PARQUET)
    df_Empr_Inconsistente = df_Empr_Bronze[df_Empr_Bronze['matricula_ou_siape'].isna()]
    #
    if len(df_Empr_Inconsistente) > 0:
        bolResult = fxSaveParquet(
        df = df_Empr_Inconsistente,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )

    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_INCONSISTENTE}.{Config.EXT_PARQUET} gravado com sucesso!!!' )
    print(f'Qtd Total de registros...........: {len(df_Empr_Inconsistente)}')

except OSError as e:
    print(e)


Arquivo: DB_Empr_Silver.parquet gravado com sucesso!!! 

Total de registros após tratamento de dados.......: 2,257,981 

Total de registros únicos após tratamento de dados: 968,028
Arquivo: DB_Empr_Inconsistente.parquet gravado com sucesso!!!
Qtd Total de registros...........: 3170


### Teste de leitura de um empréstimo realizado

In [15]:
df_Empr_Silver.loc[df_Empr_Silver['id_emprestimo']==10322328]

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario
1849850,10322328,2014070441,2018/01/15 12:26:00.697000000,2017/12/29 18:21:09.060000000,2018/01/31 18:50:25.000000000,2.014063e+09,ALUNO DE GRADUAÇÃO


## 1ª Etapa - Parte 5 - Importando o arquivo: dados_exemplares.parquet
- Importar o arquivo e gravá-lo como CSV
- Incluir seu conteúdo no arquivo: DB_Emprestimo.csv

In [16]:
# Leitura do arquivo: dados_exemplares.parquet
df_CadLivros   = pd.read_parquet(f'{Config.URL_PARQUET}')

df_CadLivros.head()

,id_exemplar,codigo_barras,colecao,biblioteca,status_material,localizacao,registro_sistema
index,,,,,,,
0,5,L000003,Acervo Circulante,Biblioteca Central Zila Mamede,REGULAR,694,1
1,4,L000002,Acervo Circulante,Biblioteca Central Zila Mamede,REGULAR,688,1
2,3,L000001,Acervo Circulante,Biblioteca Central Zila Mamede,ESPECIAL,638,1
3,7,L000114,Acervo Circulante,Biblioteca Central Zila Mamede,REGULAR,616,5
5,10,L000041,Acervo Circulante,Biblioteca Central Zila Mamede,ESPECIAL,657,15


In [17]:
print(f'Total de registros: {len(df_CadLivros):,} do arquivo: DADOS_EXEMPLARES.parquet')

Total de registros: 546,237 do arquivo: DADOS_EXEMPLARES.parquet


In [18]:
bolResult = False
try:
    strPathFile  = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_SILVER_V2, Config.EXT_PARQUET)
    df_CadLivros = pd.read_parquet(f'{Config.URL_PARQUET}')
    #
    df_Empr_Silver_v2 = df_Empr_Silver.merge(df_CadLivros)     # Faz o JOIN entre as tabelas gerando o Dataframe df_Emprestimo_Tratado

    bolResult = fxSaveParquet(
        df = df_Empr_Silver_v2,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_SILVER_V2}.{Config.EXT_PARQUET} gravado com sucesso!!! \n' )
        print(f'Total de registros antes do Merge de dados........: {len(df_Empr_Silver):,} \n')
        print(f'Total de registros após o Merge de dados..........: {len(df_Empr_Silver_v2):,} \n')
        print(f'Total de registros únicos após tratamento de dados: {len(df_Empr_Bronze.value_counts()):,}')
except OSError as e:
    print(e)

# A diferença de registros entre os DataFrames df_Empr_Silver e df_Empr_Silver_v2 é de 185.459
# Não entendi o motivo desta diferença.

Arquivo: DB_Empr_Silver_V2.parquet gravado com sucesso!!! 

Total de registros antes do Merge de dados........: 2,257,981 

Total de registros após o Merge de dados..........: 2,072,522 

Total de registros únicos após tratamento de dados: 968,028


### Análise do que ocorreu na etapa anterior

In [19]:
bolResult = False
try:
    strPathFile  = fxPathFile(Config.PATH_BASE, Config.DB_EMPR_PERDIDOS, Config.EXT_PARQUET)
    #
    intQtdRegAntesMerge = len(df_Empr_Silver)
    intQtdRegAposMerge  = len(df_Empr_Silver_v2)
    intQtdRegUnicos     = len(df_Empr_Bronze.value_counts())
    intDiferenca        = intQtdRegAntesMerge - intQtdRegAposMerge
    #
    # Método 1: Usando merge com indicator
    comparacao = df_Empr_Silver.merge(
        df_Empr_Silver_v2[['id_emprestimo']],  # Apenas a coluna chave para economizar memória
        on='id_emprestimo',
        how='left',
        indicator=True
    )
    df_Empr_Perdidos = comparacao[comparacao['_merge'] == 'left_only'].drop(columns=['_merge'])

    # Estatísticas básicas
    print(f"Total df_Empr_Silver...: {intQtdRegAntesMerge:,} \n")
    print(f"Total df_Empr_Silver_v2: {intQtdRegAposMerge:,} \n")
    print(f'Diferença..............: {intDiferenca:,} \n')
    print(f"REGISTROS PERDIDOS.....: {len(df_Empr_Perdidos):,} \n")

    # Análise dos registros perdidos
    if len(df_Empr_Perdidos) > 0:
        print(f"Amostra dos registros perdidos: \n")
        print(df_Empr_Perdidos.head(10))
        print('\n')

        # Análise por tipo de vínculo (se aplicável)
        if 'tipo_vinculo_usuario' in df_Empr_Perdidos.columns:
            print(f"Distribuição por tipo de vínculo: {df_Empr_Perdidos['tipo_vinculo_usuario'].value_counts()} \n")
            #print(df_Empr_Perdidos['tipo_vinculo_usuario'].value_counts())

        # Verificar padrões de valores nulos
        print("Valores nulos nos registros perdidos: \n")
        print(df_Empr_Perdidos.isna().sum())

    bolResult = fxSaveParquet(
        df = df_Empr_Perdidos,
        caminho = strPathFile,
        verificar_integridade = True,
        comprimir = 'snappy'
    )
    if bolResult:
        print( f'Arquivo: {Config.DB_EMPR_PERDIDOS}.{Config.EXT_PARQUET} gravado com sucesso!!! \n' )
except OSError as e:
    print(e)

Total df_Empr_Silver...: 2,257,981 

Total df_Empr_Silver_v2: 2,072,522 

Diferença..............: 185,459 

REGISTROS PERDIDOS.....: 189,257 

Amostra dos registros perdidos: 

     id_emprestimo codigo_barras                 data_renovacao  \
17          709701       L163841  2010/01/19 19:02:53.450000000   
98          709782       L165850                            NaN   
109         709793       L178388  2010/01/19 16:31:44.491000000   
118         709802    2009032201  2010/01/19 16:13:22.082000000   
119         709803       L188583  2010/02/01 15:52:28.614000000   
125         709809       L066881                            NaN   
137         709821       L154477  2010/01/19 00:04:10.604000000   
151         709835       L149054                            NaN   
152         709836       L167962                            NaN   
163         709847       L178909  2010/01/19 13:36:19.566000000   

                   data_emprestimo                 data_devolucao  \
17   2010/01/04

In [20]:
df_Empr_Perdidos.head()

,id_emprestimo,codigo_barras,data_renovacao,data_emprestimo,data_devolucao,matricula_ou_siape,tipo_vinculo_usuario
17,709701,L163841,2010/01/19 19:02:53.450000000,2010/01/04 09:48:05.604000000,2010/02/04 18:23:49.946000000,2.008029e+09,ALUNO DE GRADUAÇÃO
98,709782,L165850,NaN,2010/01/04 14:06:16.699000000,2010/01/22 16:09:20.065000000,2.006213e+08,ALUNO DE GRADUAÇÃO
109,709793,L178388,2010/01/19 16:31:44.491000000,2010/01/04 14:47:41.976000000,2010/02/01 16:29:17.951000000,2.005063e+08,ALUNO DE GRADUAÇÃO
118,709802,2009032201,2010/01/19 16:13:22.082000000,2010/01/04 14:57:23.766000000,2010/02/03 14:04:04.222000000,2.009054e+09,ALUNO DE GRADUAÇÃO
119,709803,L188583,2010/02/01 15:52:28.614000000,2010/01/04 15:03:04.753000000,2010/03/02 17:58:07.985000000,2.010117e+09,ALUNO DE PÓS-GRADUAÇÃO


In [21]:
print(len(df_Empr_Silver))
print(len(df_Empr_Silver_v2))
print(len(df_Empr_Perdidos))

2257981
2072522
189257
